In [26]:
# builtins
from pathlib import Path
import sys
import json

sys.path.append('..')

# local
from rasscol_src.general_utils import *
from rasscol_src.rasscol_utils import *

import tensorflow_decision_forests as tfdf
import tensorflow as tf

rasscol = RASSCoL()


In [27]:
with open('../example/scapCC4_NRD_RF/config.json', 'r') as f:
    config = json.load(f)

In [28]:
# generate a dataframe from the dictionary to match RP's format

# import modules
import pandas as pd

# make a dataframe from the 
df_seqs = pd.DataFrame(config['seqs']).T

# split up the pocket sequences into one residue per column
# dropping the first and last column as these will be empty strings ('')
# (N seqs x N pocket residues)
df_pocket_exploded = df_seqs['pocket_seq'].str.split('', expand=True).iloc[:,1:-1]

# rename columns
df_pocket_exploded.columns = [f's{n}' for n in range(df_pocket_exploded.shape[1])]

# split up layer volumes into columns (N seqs x N layers)
vol_cols = [f'v{n}' for n in range(len(df_seqs.loc['0','volume']))]
df_volume_exploded = pd.DataFrame(df_seqs.volume.tolist(), index=df_seqs.index, columns=vol_cols)

# join the sequence and volume dataframes (N seqs x (N layers + N pocket residues))
df_rf = pd.concat([df_pocket_exploded, df_volume_exploded], axis=1)

df_rf['Docking'] = 0.

In [29]:
import random
seed = 42
random.seed(seed)

def sample_and_remove(numbers:list, number_to_sample) -> list:
    random_numbers = []
    for _ in range(number_to_sample):
        random_number = random.choice(numbers)
        random_numbers.append(random_number)
        numbers.remove(random_number)
        
    return numbers, random_numbers

In [30]:
import numpy as np

chain='A'
all_seqs = df_rf
wd=Path('../example/scapCC4_NRD_RF')
step_size=500

ligand=config['run']['ligand_path']
scaffold=config['run']['receptor_path']

sequence=pdb2seq(scaffold)[chain]
pos = []
for x in config['design']:
    for res_num in config['design'][x]['res_info']:
        pos.append(int(res_num))
centroid=config['run']['pocket_ca_centroid']

cores=config['run']['num_cpus']

#shuffle the dataset
seq_ids = list(all_seqs.index)

seq_ids, train_index = sample_and_remove(seq_ids, round(step_size*0.8))
seq_ids, mse_index = sample_and_remove(seq_ids, round(step_size*0.2))

rasscol.run_parallel(sequence, pos, config, [str(x) for x in train_index])
rasscol.run_parallel(sequence, pos, config, [str(x) for x in mse_index])

Job 841368 (SAVTAVISA) finished with vina_score (norm): 0.897 (0.037)
Job 13628 (GALAAVVAA) finished with vina_score (norm): -2.352 (-0.098)
Job 551057 (LGSGITSGI) finished with vina_score (norm): -6.079 (-0.253)
Job 403276 (ASTTASLAA) finished with vina_score (norm): -4.494 (-0.187)
Job 233478 (AAISVSVSS) finished with vina_score (norm): -7.411 (-0.309)
Job 800397 (SGVAIATLG) finished with vina_score (norm): 0.139 (0.006)
Job 1189546 (TTSTGTGTI) finished with vina_score (norm): 0.507 (0.021)
Job 963519 (SSTSLASST) finished with vina_score (norm): 0.43 (0.018)
Job 468108 (IGTSGLGSL) finished with vina_score (norm): 0.683 (0.028)
Job 55650 (GITTSSGAI) finished with vina_score (norm): -3.664 (-0.153)
Job 566161 (LGTIGTGTL) finished with vina_score (norm): 1.177 (0.049)
Job 292850 (ALGGLSSLG) finished with vina_score (norm): 0.108 (0.005)
Job 459986 (IGSTVALAA) finished with vina_score (norm): -3.108 (-0.13)
Job 341132 (AVSATTAIA) finished with vina_score (norm): 1.456 (0.061)
Job 117317 

In [31]:
results_df = pd.read_csv('/home/jc17773/workspace/RASSCoL_no_RF/example/scapCC4_NRD_RF/RASSCoL_results.csv', index_col='id')
results_df.index = results_df.index.astype(str)

for idx in [train_index, mse_index]:
    all_seqs.loc[idx,'Docking'] = results_df.loc[idx, 'vina_score']

In [32]:
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_decision_forests as tfdf

# ------------------------- Script Settings -------------------------

steps = 5  # Number of active learning steps
step_size = 500
stopping_patience = None  # Controls early stopping, set to an integer for validation patience
verbose = 0  # 0 for minimal output, 1 for detailed output
num_models = 10  # Number of models to train in the ensemble

# ------------------------- Warnings and Initialization -------------------------

mse_list = []

if stopping_patience is None:
    warnings.warn('stopping_patience=None, no validation set will be merged with the training set')

# ------------------------- Main Loop -------------------------

for i in range(steps):
    print(f'\nStarting Step {i + 1}/{steps}')

    # Prepare training and test datasets
    if stopping_patience is not None:
        df_train = all_seqs.loc[train_index]
        df_mse = all_seqs.loc[mse_index]
    else:
        df_train = all_seqs.loc[np.concatenate((train_index, mse_index))]
    
    df_test = all_seqs.drop(np.concatenate((train_index, mse_index)))
    df_mse = all_seqs.loc[mse_index]
    
    # Drop the "Predictions" column if it exists
    features = [col for col in df_train.columns if col not in ['Docking', 'Predictions', 'Predictions_std']]

    # Convert data to TensorFlow datasets
    train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(df_train[features + ['Docking']], label='Docking', task=tfdf.keras.Task.REGRESSION)
    test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(df_test[features + ['Docking']], label='Docking', task=tfdf.keras.Task.REGRESSION)
    mse_ds = tfdf.keras.pd_dataframe_to_tf_dataset(df_mse[features + ['Docking']], label='Docking', task=tfdf.keras.Task.REGRESSION)

    # ------------------------- Model Training -------------------------

    print('Training Gradient Boosted Trees models...')
    tuner = tfdf.tuner.RandomSearch(num_trials=20, use_predefined_hps=True)
    models = []
    
    for k in range(num_models):
        model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner, task=tfdf.keras.Task.REGRESSION, random_seed=k, verbose=verbose)
        model.fit(train_ds)
        models.append(model)
        print(f'Model {k + 1}/{num_models} trained.')

    print(f'{num_models} models trained successfully.')

    # ------------------------- Combine Models for Ensemble -------------------------

    class CombinedModel(tf.keras.Model):
        """Ensemble model combining predictions from multiple Gradient Boosted Trees models."""
        def call(self, inputs):
            predictions = tf.concat([submodel(inputs) for submodel in models], axis=1)
            return tf.math.reduce_mean(predictions, axis=1), tf.math.reduce_std(predictions, axis=1)

    combined_model = CombinedModel()

    # ------------------------- Linear Scaling for Predictions -------------------------

    print('Calibrating model predictions to align with experimental docking scores...')
    train_predictions, _ = combined_model.predict(train_ds)
    coef = np.polyfit(train_predictions.flatten(), df_train['Docking'].values, 1)
    scaling_function = np.poly1d(coef)

    # ------------------------- Evaluate on MSE Dataset -------------------------

    if stopping_patience is not None:
        mse_predictions, _ = combined_model.predict(mse_ds)
        mse_predictions_scaled = scaling_function(mse_predictions.flatten())
        mse = np.mean(np.square(mse_predictions_scaled - df_mse['Docking'].values))

        mse_list.append(mse)
        print(f'Step {i + 1} MSE: {mse:.4f}')

        # Check for early stopping
        if i >= stopping_patience and all(x > mse for x in mse_list[-stopping_patience:]):
            print(f'Stopping early after {i + 1} steps due to no improvement in MSE.')
            break

    # ------------------------- Active Sampling -------------------------

    test_predictions, prediction_std = combined_model.predict(test_ds)
    test_predictions_scaled = scaling_function(test_predictions.flatten())

    # Save predictions
    all_seqs.loc[df_test.index, 'Predictions'] = test_predictions_scaled
    all_seqs.loc[df_test.index, 'Predictions_std'] = prediction_std

    # Select samples for the next round
    worst_index = all_seqs.nlargest(round(step_size * 0.25), 'Predictions_std').index  # High uncertainty
    best_index = all_seqs.nsmallest(round(step_size * 0.25), 'Predictions').index  # Best docking scores

    # Combine worst and best indices, ensuring no duplicates
    combined_index = worst_index.union(best_index)

    # Exclude indices already in df_train to avoid duplicates
    allowed_sample_space = all_seqs.index.difference(df_train.index.union(combined_index))

    # Randomly sample from the allowed sample space
    random_index = allowed_sample_space.to_series().sample(n=round(step_size * 0.5), random_state=seed).index

    # Final combined index with worst, best, and random samples
    final_combined_index  = pd.Index(worst_index).union(best_index).union(random_index).unique()
    new_test_index = list(final_combined_index.difference(results_df.index))

    # ------------------------- Dock New Samples -------------------------

    print(f'Docking {len(new_test_index)} new samples...')
    rasscol.run_parallel(sequence, pos, config, [str(x) for x in new_test_index])
    results_df = pd.read_csv('/home/jc17773/workspace/RASSCoL_no_RF/example/scapCC4_NRD_RF/RASSCoL_results.csv', index_col='id')
    results_df.index = results_df.index.astype(str)

    # Update training data with new docking scores
    all_seqs.loc[new_test_index, 'Docking'] = results_df.loc[new_test_index, 'vina_score']
    train_index = list(set(train_index + new_test_index))

print('\nActive learning process completed.')


/tmp/ipykernel_175291/2752858714.py:20: UserWarning: stopping_patience=None, no validation set will be merged with the training set
  warnings.warn('stopping_patience=None, no validation set will be merged with the training set')



Starting Step 1/5
Training Gradient Boosted Trees models...


[WARNING 24-12-19 17:46:17.7050 GMT gradient_boosted_trees.cc:1840] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:46:17.7050 GMT gradient_boosted_trees.cc:1851] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:46:17.7050 GMT gradient_boosted_trees.cc:1865] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
[INFO 24-12-19 17:46:19.3406 GMT kernel.cc:1233] Loading model from path /tmp/tmpmixbhh65/model/ with prefix 2300c9cd61b44fec
[INFO 24-12-19 17:46:19.3418 GMT decision_forest.cc:734] Model loaded with 19 root(s), 381 node(s), and 12 input feature(s).
[INFO 24-12-19 17:46:19.3418 GMT abstract_model.cc:1344] Engine "GradientBoostedTreesGeneric" built
[INFO 24-12-19 17:46:19.3419 GMT kernel.cc:1061] Use fast generic engine
[WARNING 24-12-19 17:46:19.5183 GMT gradient_boosted_trees.cc:1840] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:46:19.5183 GMT gradi

Model 1/10 trained.


[INFO 24-12-19 17:46:21.2656 GMT kernel.cc:1233] Loading model from path /tmp/tmpzb8943s1/model/ with prefix 533228a601a8424c
[INFO 24-12-19 17:46:21.2667 GMT decision_forest.cc:734] Model loaded with 39 root(s), 455 node(s), and 12 input feature(s).
[INFO 24-12-19 17:46:21.2667 GMT kernel.cc:1061] Use fast generic engine
[WARNING 24-12-19 17:46:21.4380 GMT gradient_boosted_trees.cc:1840] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:46:21.4380 GMT gradient_boosted_trees.cc:1851] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:46:21.4380 GMT gradient_boosted_trees.cc:1865] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


...

Model 8/10 trained.


[INFO 24-12-19 17:52:46.5174 GMT kernel.cc:1233] Loading model from path /tmp/tmpj7cj76eu/model/ with prefix c75bc88f7aeb4868
[INFO 24-12-19 17:52:46.5246 GMT decision_forest.cc:734] Model loaded with 50 root(s), 4344 node(s), and 12 input feature(s).
[INFO 24-12-19 17:52:46.5246 GMT abstract_model.cc:1344] Engine "GradientBoostedTreesGeneric" built
[INFO 24-12-19 17:52:46.5246 GMT kernel.cc:1061] Use fast generic engine
[WARNING 24-12-19 17:52:46.7171 GMT gradient_boosted_trees.cc:1840] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:52:46.7171 GMT gradient_boosted_trees.cc:1851] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 24-12-19 17:52:46.7171 GMT gradient_boosted_trees.cc:1865] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".


Model 9/10 trained.


[INFO 24-12-19 17:52:52.8366 GMT kernel.cc:1233] Loading model from path /tmp/tmp73dw4ftl/model/ with prefix 735343ff4dc94549
[INFO 24-12-19 17:52:52.8406 GMT decision_forest.cc:734] Model loaded with 36 root(s), 2682 node(s), and 12 input feature(s).
[INFO 24-12-19 17:52:52.8406 GMT kernel.cc:1061] Use fast generic engine


Model 10/10 trained.
10 models trained successfully.
Calibrating model predictions to align with experimental docking scores...
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 250ms/step
1190/1190 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step
Docking 250 new samples...
Job 217242 (AGLASTLGA) finished with vina_score (norm): -6.01 (-0.25)
Job 1026861 (TGVIGTIGA) finished with vina_score (norm): 1.606 (0.067)
Job 1064961 (TASSVSSST) finished with vina_score (norm): 0.415 (0.017)
Job 163945 (GTIVATSGI) finished with vina_score (norm): -5.288 (-0.22)
Job 239650 (AALASVTGL) finished with vina_score (norm): 2.208 (0.092)
Job 113540 (GVTGSIGTI) finished with vina_score (norm): -2.962 (-0.123)
Job 1043260 (TGTTGLGVV) finished with vina_score (norm): 1.54 (0.064)
Job 1039549 (TGTLSAGLT) finished with vina_score (norm): -1.114 (-0.046)
Job 201842 (GTTTSTAGI) finished with vina_score (norm): 0.213 (0.009)
Job 107868 (GVSVVGVSS) finished with vina_score (norm): 1.917 (0.08)
Job 1016688 (TGLLAGIAG) finished with vina_score (norm